# 1.6 Lab: CLI Assistant with Memory

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Prerequisites

Before this lab, work through:

- **0.2** for the virtual environment, `.env` file, and `OPENAI_API_KEY`.
- **0.4** for system prompts and few-shot prompting (you will write the assistant's system prompt).
- **0.5** for your first structured-output classifier (this lab adds a second one, inside a real app).
- **1.1** for the `|` pipe operator and `RunnablePassthrough.assign()` (the trimming step uses both).
- **1.3** for Pydantic v2 schemas and `with_structured_output` (the intent tagger).
- **1.5** for `RunnableWithMessageHistory`, session ids, and why the buffer needs a window.

## Learning objectives

- Build a working terminal chatbot that remembers the conversation using `RunnableWithMessageHistory` with a windowed history (no legacy memory classes).
- Trim the injected history to a token budget with `trim_messages`, keeping the cost of each turn flat.
- Add a side feature with provider-native structured output: classify the user's intent on every turn with a Pydantic model.
- Handle the three failure classes a CLI assistant meets in practice: missing or bad API key, rate limits and network errors, and empty input.
- Explain the factory-copy trap: why trimming must happen in the chain, not by slicing inside the history factory.

## What you are building

A single-file program, `cli_assistant.py`, that runs a chatbot in your terminal. Each turn it does two model calls: one classifies the user's intent into a Pydantic object (a visible tag, like `[intent: question]`), and one produces the reply with the recent conversation injected from memory. The history is windowed to a token budget, so a hundred-turn session costs the same per reply as a ten-turn one.

One honest note before you run it. In the pinned versions, importing and using `RunnableWithMessageHistory` prints a `LangChainDeprecationWarning` saying it is deprecated in favour of LangGraph persistence. It still works exactly as lesson 1.5 describes, and this lab is the right place to learn the message-list mechanics. Module 4 teaches the LangGraph checkpointer approach the warning points to. Warnings are information, not errors; do not "fix" this one by downgrading packages.

## Step 1: the history store and the session factory

The store is a plain dict of session id to message list, exactly as in lesson 1.5. The lab adds one rule on top: the factory returns the *real* store object, and the windowing happens later, inside the chain.

In [ ]:
"""cli_assistant.py - a terminal chatbot with windowed memory and intent tagging.

Setup (run once):
    python -m venv .venv && source .venv/bin/activate
    pip install "langchain==1.4.6" "langchain-core==1.6.12" \
        "langchain-openai==1.7.2" "python-dotenv" "pydantic>=2"
    printf 'OPENAI_API_KEY=sk-...\n' > .env

Run:
    python cli_assistant.py
"""

import os
import sys
import time

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
import openai

load_dotenv()

SYSTEM_PROMPT = (
    "You are a concise study assistant for a programming course. "
    "Answer in plain British English, keep replies under 120 words, "
    "and use short code snippets where they help. If you do not know "
    "something, say so instead of guessing."
)
HISTORY_BUDGET_TOKENS = 3000   # max history tokens sent per turn
SESSION_ID = "cli-user-1"


class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: list[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: list[BaseMessage]) -> None:
        self.messages.extend(messages)

    def clear(self) -> None:
        self.messages = []


store: dict[str, InMemoryHistory] = {}


def get_session_history(session_id: str) -> InMemoryHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]

Read that factory twice. It returns the live store object, never a slice of it. There is a trap here that catches almost everyone once: if the factory returns `full.messages[-10:]` as a *new* history object, the wrapper appends the new turn's messages to that throwaway copy, and the real store never grows. The assistant then "forgets" everything every turn. Trimming belongs in the chain, where the wrapper has already loaded the messages. The "What can go wrong" section returns to this.

## Step 2: the chat chain with a trimming step

The chain takes `{"input": user_text}`, trims the loaded history to the token budget, renders the prompt, and calls the model. The trim uses `trim_messages` from `langchain_core.messages` with `token_counter="approximate"` (about four characters per token, good enough for a budget) and `start_on="human"` so the window never starts mid-exchange with a dangling AI reply.

In [ ]:
def trim_to_budget(history: list[BaseMessage]) -> list[BaseMessage]:
    """Keep the most recent history inside the token budget."""
    return trim_messages(
        history,
        max_tokens=HISTORY_BUDGET_TOKENS,
        strategy="last",
        token_counter="approximate",
        start_on="human",
    )


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

chat_chain = (
    RunnablePassthrough.assign(history=lambda x: trim_to_budget(x["history"]))
    | prompt
    | chat_model
)

chat_with_memory = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": SESSION_ID}}

The `RunnablePassthrough.assign(history=...)` step is the windowing. The wrapper injects the full stored history under the `"history"` key; the assign step replaces it with the trimmed version before the prompt renders. The store keeps every message (useful later for summaries or analytics), while the model only ever sees the budget. Cost per turn stays flat no matter how long the session runs.

## Step 3: the intent tagger (structured output side feature)

Every turn also classifies the user's message with provider-native structured output, the `with_structured_output` route from lesson 1.3. The tag is printed next to the reply: a small feature, but it is exactly the shape real products use (routing, analytics, guardrails all start as "classify each turn").

In [ ]:
class IntentTag(BaseModel):
    intent: Literal["question", "smalltalk", "task", "complaint", "other"] = Field(
        description="The user's intent in this one message."
    )
    needs_history: bool = Field(
        description="True if answering well needs earlier conversation context, False for standalone messages."
    )


tagger = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(IntentTag)


def classify(text: str) -> IntentTag:
    return tagger.invoke(f"Classify this user message. Message: {text!r}")

Note the cost honestly: this doubles the model calls per turn. At `gpt-4o-mini` prices that is still tiny for a lab, but in production you would batch this, sample it (classify every Nth turn), or fold it into the main call. The lab keeps it per-turn so you can watch it work.

## Step 4: the main loop with graceful errors

The loop handles four things: empty input (skip, do not send), `quit`/`Ctrl-C`/`Ctrl-D` (clean exit), API failures (named, actionable messages), and anything else (print and keep going, never crash the session).

In [ ]:
def check_api_key() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print(
            "OPENAI_API_KEY is not set.\n"
            "Create a .env file next to cli_assistant.py with:\n"
            "    OPENAI_API_KEY=sk-...\n"
            "Then run the program again."
        )
        sys.exit(1)


def main() -> None:
    check_api_key()
    print("Assistant ready. Type 'quit' to exit.\n")

    while True:
        try:
            user_text = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye.")
            break

        if not user_text:
            continue
        if user_text.lower() in {"quit", "exit"}:
            print("Goodbye.")
            break

        try:
            tag = classify(user_text)
            print(f"  [intent: {tag.intent} | needs_history: {tag.needs_history}]")
            reply = chat_with_memory.invoke({"input": user_text}, config=config)
            print(f"Assistant: {reply.content}\n")

        except openai.AuthenticationError:
            print(
                "Your API key was rejected by OpenAI. It may be wrong, "
                "revoked, or pasted with a typo. Fix OPENAI_API_KEY and restart."
            )
            sys.exit(1)
        except openai.RateLimitError:
            print(
                "Rate limit hit. Waiting 20 seconds before you try your next "
                "message. If this keeps happening, add delays between turns."
            )
            time.sleep(20)
        except openai.APIConnectionError:
            print(
                "Could not reach the OpenAI API. Check your connection and "
                "try the message again."
            )
        except Exception as exc:  # keep the session alive for anything else
            print(f"Something went wrong ({type(exc).__name__}): {exc}")
            print("Your history is intact. Try the message again.")


if __name__ == "__main__":
    main()

A detail worth knowing: `langchain-openai` converts provider errors into its own subclasses, but those subclasses *also* inherit from the `openai` error classes (`OpenAIAuthenticationError` is an `openai.AuthenticationError`). Catching the `openai` names therefore works whether the error comes from the raw client or the converted one, and the same names work if you later switch to another OpenAI-compatible provider.

## How to run

In [ ]:
!python -m venv .venv && source .venv/bin/activate
!pip install "langchain==1.4.6" "langchain-core==1.6.12" \
!    "langchain-openai==1.7.2" "python-dotenv" "pydantic>=2"
!printf 'OPENAI_API_KEY=sk-...\n' > .env
!python cli_assistant.py

Expect a `LangChainDeprecationWarning` about `RunnableWithMessageHistory` on startup (explained above), then the prompt. Try this sequence to exercise every feature: tell it your name, ask an unrelated question, then ask it to recall your name (memory), then paste a long multi-paragraph message twice (windowing keeps working), then try an empty line and `quit`.

## Sample session

What a healthy run looks like (your wording will differ; the structure should match):

In [ ]:
Assistant ready. Type 'quit' to exit.

You: My name is Saud and I am learning Python decorators.
  [intent: smalltalk | needs_history: False]
Assistant: Nice to meet you, Saud. Decorators are a good topic to pick...

You: What is my name?
  [intent: question | needs_history: True]
Assistant: Your name is Saud.

You:
You: quit
Goodbye.

Notice three things: the empty line was skipped without a model call, the name recall proves history injection works, and the tagger correctly flagged the recall question as needing history. If the name recall fails, check the `MessagesPlaceholder` name against `history_messages_key` first (lesson 1.5's checklist).

## Stretch goals

Pick at least one. Each is a half-hour task that points at a later module.

1. **Persist the history to disk.** The dict store dies with the process. On exit, write each session's messages to a JSON file with `message.dict()`; on startup, load them back with `langchain_core.messages` deserialisation. This is the smallest possible step toward production memory, and it makes the GDPR retention question from lesson 1.5 concrete: add a `--forget` flag that deletes a session file.
2. **Add a `/summary` command.** When the user types `/summary`, take the full stored history, ask the model to compress everything older than the last 4 messages into one paragraph, and store that paragraph as a `SystemMessage` at the front of the history. You have just built summary memory from lesson 1.5 by hand.
3. **Track open tasks as a second side feature.** Define a `TaskList` Pydantic model (`tasks: list[str]`) and, alongside the intent tag, extract "things the user said they will do" each turn. Print the running list after each reply. This is the same structured-output muscle, pointed at state the assistant maintains about the user: a preview of the stateful agents in Module 4.

## What can go wrong

- **The factory-copy trap.** Returning a sliced copy from `get_session_history` means the wrapper appends new turns to the copy and the real store never grows. Symptom: the assistant forgets everything every turn, while `len(store[...].messages)` stays flat. Fix: return the live store object; trim in the chain with `RunnablePassthrough.assign`, as this lab does.
- **The deprecation warning.** `RunnableWithMessageHistory` warns that it is deprecated in favour of LangGraph persistence. It is not broken in the pinned versions. Do not silence it with warning filters in a way that hides future breakage; read it as "learn the LangGraph way in Module 4".
- **Approximate token counting.** `token_counter="approximate"` assumes roughly four characters per token, which undercounts for code, non-English text, and long numbers. If your sessions carry lots of code, the real token count can overshoot the budget. For a lab it is fine; production code passes the model's own counter or a tokenizer.
- **The tagger doubles your calls.** Two model calls per turn means twice the latency and twice the input tokens. Watch for this in the "needs_history" tag: if it is almost always True, the tag is not discriminating and you are paying for noise. Tune the field descriptions.
- **`start_on="human"` and odd windows.** Without it, a token budget can cut the window so it starts with an AI message that answered a question the model can no longer see. The replies then read as non sequiturs. Always pin the window start to a human turn.
- **Rate limits on long sessions.** The window keeps input tokens flat, but output tokens and the tagger calls still add up. A `429` mid-session is handled by the loop's wait-and-continue; a `401` (bad key) exits, because retrying a rejected key is pointless.
- **Shared session ids.** `SESSION_ID` is a constant here, fine for one person at one terminal. In anything multi-user, the session id must come from the authenticated user, or strangers share one memory.
- **History is personal data.** This lab stores everything the user typed in a process dict. Lesson 1.5's GDPR note applies the moment this leaves your laptop: scope per user, set retention, support deletion.

## Check your understanding

**1. Why does the trimming step sit in the chain (`RunnablePassthrough.assign`) instead of inside `get_session_history`?**
Because the wrapper appends the new turn's messages to whatever object the factory returns. If the factory returns a sliced copy, the appends land on the copy and the real store never grows, so the assistant forgets every turn. Trimming in the chain keeps the store complete and only narrows what the model sees.

**2. What does `start_on="human"` do in `trim_messages`, and what breaks without it?**
It forces the trimmed window to begin with a human message. Without it, a token budget can slice the history so it starts with an AI reply whose question was cut away, and the model answers from a conversation fragment that reads as nonsense.

**3. Your session hits a `429` rate limit error. The loop waits 20 seconds and continues. Why does a `401` authentication error exit instead?**
A rate limit is transient: waiting and retrying can succeed. A rejected API key will never succeed on retry, so the only fix is outside the program (correct the key). Exiting with a clear message beats an infinite retry loop.

**4. The intent tagger runs on every turn. Name two production concerns with that design.**
Cost and latency double (two model calls per turn, and the tagger re-sends the message), and if the tag is almost always the same value it is noise you are paying for. Production options: sample every Nth turn, or fold the classification into the main call.

**5. Where does the deprecation warning for `RunnableWithMessageHistory` point you, and why is this lab still the right place to learn it?**
It points to LangGraph's built-in persistence (checkpointers), taught in Module 4. The lab is still right because the underlying mechanics, a message list loaded before the call and saved after, are identical there; LangGraph just manages the list inside graph state instead of a wrapper.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.